# Lab 4 — RL Training with Isaac Lab

Train robot policies in simulation using reinforcement learning.
**Run `Lab4_0_Build_Container.ipynb` first** to ensure the Isaac Lab image is in ECR.

---

## Prerequisites (read before running)

| Requirement | Details |
|-------------|---------|
| **Isaac Lab container in ECR** | Run `Lab4_0_Build_Container.ipynb` first |
| **GPU instance quota** | `ml.g5.xlarge` (1× A10G) — G-family only; P-family lacks RT Cores and will crash |
| **SageMaker role** | Standard SageMaker + S3 permissions — same role as Lab 1 |
| **DATASETS_BUCKET env var** | Set before calling `launch_rl.py` (Section 1 does this automatically) |

## What this notebook does

| Section | Task | Validated |
|---------|------|-----------|
| 1. Smoke test | `Isaac-Velocity-Flat-Anymal-D-v0` — Anymal quadruped walking (50 iters) | |
| 2. UR10 arm reach | `Isaac-Reach-UR10-v0` — UR10 arm reaching task (100 iters) | |
| 3. Monitor | Poll status for either job | |
| 4. Download results | Get checkpoint + metadata locally | |
| 5. Render Anymal-D video | MP4 of quadruped policy | |
| 6. Render UR10 video | MP4 of arm reaching policy | |
| 7. Summary | What worked, what's next | — |

## Honest status

> **Both validated tasks use Isaac Lab built-in environments** — no custom USD assets needed.
>
> **`PickAndPlaceUR3-v0`** is the target task but is **not yet wired into the container**
> by design — it is planned future work.
>
> The GR00T→MLP bridge (`groot_to_rl_bridge.py`) is also future work — today's
> RL runs are standalone (no Lab 1 dependency required).


## 0 — Setup

In [ ]:
import boto3, json, os, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'aws-physical-ai-toolchain' and REPO_ROOT != REPO_ROOT.parent:
 REPO_ROOT = REPO_ROOT.parent

REGION = boto3.session.Session().region_name or 'us-west-2'
ACCOUNT = boto3.client('sts', region_name=REGION).get_caller_identity()['Account']
BUCKET = f'sagemaker-{REGION}-{ACCOUNT}'
ROLE_ARN = f'arn:aws:iam::{ACCOUNT}:role/service-role/AmazonSageMaker-ExecutionRole-20260607T114524'
ECR_URI = f'{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/physical-ai/isaac-lab:latest'

sm = boto3.client('sagemaker', region_name=REGION)
s3 = boto3.client('s3', region_name=REGION)
ecr = boto3.client('ecr', region_name=REGION)

print(f'Region: {REGION}')
print(f'Account: {ACCOUNT}')
print(f'Bucket: {BUCKET}')
print(f'ECR: {ECR_URI}')


In [ ]:
# Verify Isaac Lab container is available
try:
 imgs = ecr.describe_images(
 repositoryName='physical-ai/isaac-lab',
 imageIds=[{'imageTag': 'latest'}]
 )['imageDetails']
 img = imgs[0]
 print(f" Isaac Lab image: {img['imageSizeInBytes']/1e9:.1f} GB, "
 f"pushed {img['imagePushedAt'].strftime('%Y-%m-%d')}")
except Exception as e:
 print(f' Isaac Lab image not found in ECR: {e}')
 print(' Run Lab4_0_Build_Container.ipynb first.')

# Check for Lab 1 model (optional — RL runs standalone without it)
GROOT_MODEL_ARN = None
try:
 packages = sm.list_model_packages(
 ModelPackageGroupName='groot-models',
 SortBy='CreationTime', SortOrder='Descending', MaxResults=1
 )
 if packages['ModelPackageSummaryList']:
 pkg = packages['ModelPackageSummaryList'][0]
 GROOT_MODEL_ARN = pkg['ModelPackageArn']
 print(f" Lab 1 model: {pkg['ModelPackageArn'].split('/')[-1]}")
 else:
 print(' No model in groot-models registry (OK — RL runs standalone)')
except Exception as e:
 print(f' Could not check model registry: {e} (OK — RL runs standalone)')


## 1 — Smoke Test: Anymal-D Quadruped Locomotion

Run `Isaac-Velocity-Flat-Anymal-D-v0` — the Anymal-D quadruped learning to walk.
50 iterations, 128 environments, single A10G. Completes in ~8 min.

This validates the container, the SageMaker integration, and Isaac Sim startup
before moving on to the UR10 arm task.

**Validated result:** reward -0.36 → +8.58, 3741 steps/sec.

> **Instance type:** Use G-family only (`ml.g5.*`). P-family (P4, P5) lacks
> RT Cores and will crash Isaac Sim.

In [ ]:
import sys, os
sys.path.insert(0, str(REPO_ROOT))

# DATASETS_BUCKET must be set before importing launch_rl — it reads os.environ
os.environ['AWS_DEFAULT_REGION'] = REGION
os.environ['SAGEMAKER_ROLE_ARN'] = ROLE_ARN
os.environ['ISAAC_LAB_IMAGE'] = ECR_URI
os.environ['DATASETS_BUCKET'] = BUCKET # critical: launch_rl._bucket() reads this

from training.scripts.launch_rl import launch

ANYMAL_JOB = launch(
 task='Isaac-Velocity-Flat-Anymal-D-v0',
 num_envs=128, # small for smoke test — increase to 4096 for real training
 max_iterations=50, # 50 iters ≈ 8 min on ml.g5.xlarge
 framework='rsl_rl',
 instance_type='ml.g5.xlarge',
 instance_count=1,
 runtime_min=30,
 dry_run=False,
)
print()
print('Re-run the poll cell below every 2 min.')


In [ ]:
# Re-run to refresh
desc = sm.describe_training_job(TrainingJobName=ANYMAL_JOB)
status = desc['TrainingJobStatus']
secondary = desc.get('SecondaryStatus', '')
print(f'Job: {ANYMAL_JOB}')
print(f'Status: {status} / {secondary}')

if status == 'Completed':
 duration = (desc['TrainingEndTime'] - desc['TrainingStartTime']).seconds // 60
 ANYMAL_MODEL_S3 = desc['ModelArtifacts']['S3ModelArtifacts']
 print(f'\n Smoke test passed in {duration} min!')
 print(f' Isaac Lab container + SageMaker integration confirmed.')
 print(f' Model: {ANYMAL_MODEL_S3}')
elif status == 'Failed':
 print(f"\n Failed: {desc.get('FailureReason', '')[:300]}")
 print('\nCheck CloudWatch logs for common causes.')
 print(f"\nLogs: https://console.aws.amazon.com/cloudwatch/home?region={REGION}"
 f"#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs"
 f"/log-events/{ANYMAL_JOB}")
else:
 print('\n(Re-run in 2 min)')


## 2 — UR10 Arm Reaching

Run `Isaac-Reach-UR10-v0` — a UR10 robot arm learning to reach target positions.
This is the closest validated built-in task to our UR3 pick-and-place target.

100 iterations, 4096 environments, single A10G. Completes in ~10-15 min.

**Why UR10 instead of UR3?**
Isaac Lab has a built-in reaching task for UR10 (`Isaac-Reach-UR10-v0`) that runs
today with no custom USD assets. The UR3 pick-and-place task (`PickAndPlaceUR3-v0`)
is written in `training/envs/pick_and_place_ur3.py` but is **not yet wired into the
container** — this is planned future work.

In [ ]:
UR10_JOB = launch(
 task='Isaac-Reach-UR10-v0',
 num_envs=4096,
 max_iterations=100, # 100 iters ≈ 10-15 min on ml.g5.xlarge
 framework='rsl_rl',
 instance_type='ml.g5.xlarge',
 instance_count=1,
 runtime_min=60,
 dry_run=False,
)
print()
print('Re-run the poll cell below every 2-3 min.')


In [ ]:
# Re-run to refresh
desc = sm.describe_training_job(TrainingJobName=UR10_JOB)
status = desc['TrainingJobStatus']
secondary = desc.get('SecondaryStatus', '')
print(f'Job: {UR10_JOB}')
print(f'Status: {status} / {secondary}')

if status == 'Completed':
 duration = (desc['TrainingEndTime'] - desc['TrainingStartTime']).seconds // 60
 UR10_MODEL_S3 = desc['ModelArtifacts']['S3ModelArtifacts']
 print(f'\n UR10 training complete in {duration} min!')
 print(f' Model: {UR10_MODEL_S3}')
elif status == 'Failed':
 print(f"\n Failed: {desc.get('FailureReason', '')[:300]}")
 print(f"\nLogs: https://console.aws.amazon.com/cloudwatch/home?region={REGION}"
 f"#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs"
 f"/log-events/{UR10_JOB}")
else:
 print('\n(Re-run in 2-3 min)')
 print()
 print('What to look for in CloudWatch logs:')
 print(' Learning iteration N/100')
 print(' Computation: XXXX steps/s')
 print(' Mean reward: X.XX ← should increase over iterations')


## 3 — Monitor Either Job

Use this cell to monitor any job by name. Re-run to refresh.

In [ ]:
# Set to the job you want to monitor
MONITOR_JOB = ANYMAL_JOB # or UR10_JOB

desc = sm.describe_training_job(TrainingJobName=MONITOR_JOB)
status = desc['TrainingJobStatus']
secondary = desc.get('SecondaryStatus', '')

print(f'Job: {MONITOR_JOB}')
print(f'Status: {status} / {secondary}')

if status in ('Completed', 'Failed'):
 start = desc.get('TrainingStartTime')
 end = desc.get('TrainingEndTime')
 if start and end:
 print(f'Duration: {(end - start).seconds // 60} min')
 if status == 'Completed':
 print(f"Model: {desc['ModelArtifacts']['S3ModelArtifacts']}")
 else:
 print(f"Failure: {desc.get('FailureReason', '')[:300]}")
else:
 print()
 print(f'CloudWatch logs:')
 print(f' https://console.aws.amazon.com/cloudwatch/home?region={REGION}'
 f'#logsV2:log-groups/log-group/$252Faws$252Fsagemaker$252FTrainingJobs'
 f'/log-events/{MONITOR_JOB}')


## 4 — Download & Inspect Results

Downloads the model artifact (checkpoint + TensorBoard events) from S3.
Run after either job completes. Set `DOWNLOAD_JOB` to the job you want to inspect.

In [ ]:
import shutil, subprocess

# Change to UR10_JOB to inspect the UR10 arm results
DOWNLOAD_JOB = ANYMAL_JOB

desc = sm.describe_training_job(TrainingJobName=DOWNLOAD_JOB)
assert desc['TrainingJobStatus'] == 'Completed', \
 f"Job not complete yet: {desc['TrainingJobStatus']}"

MODEL_S3 = desc['ModelArtifacts']['S3ModelArtifacts']
OUT_DIR = Path.home() / f'isaac-lab-results-{DOWNLOAD_JOB.split("-")[-1]}'
if OUT_DIR.exists():
 shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir()

tar_path = OUT_DIR / 'model.tar.gz'
bucket_name = MODEL_S3.split('/')[2]
key = '/'.join(MODEL_S3.split('/')[3:])

print(f'Downloading {MODEL_S3}...')
s3.download_file(bucket_name, key, str(tar_path))
print(f'Downloaded: {tar_path.stat().st_size / 1e6:.0f} MB')

subprocess.run(['tar', '-xzf', str(tar_path), '-C', str(OUT_DIR)], check=True)

print(f'\nExtracted to: {OUT_DIR}')
print('Contents:')
for f in sorted(OUT_DIR.rglob('*'))[:20]:
 if f.is_file():
 print(f' {f.relative_to(OUT_DIR)} ({f.stat().st_size/1e3:.0f} KB)')


## 5 — Render Anymal-D Video

Renders an MP4 of the Anymal-D quadruped walking policy using Isaac Lab's
VideoRecorder. Runs as a short SageMaker job (~5-10 min) — no local GPU needed.

**Requires:** Section 1 completed (`ANYMAL_MODEL_S3` set).

In [ ]:
# Requires: ANYMAL_MODEL_S3 from Section 1 poll cell
ANYMAL_VIDEO_JOB = f'isaac-lab-video-anymal-{int(time.time())}'

sm.create_training_job(
 TrainingJobName=ANYMAL_VIDEO_JOB,
 RoleArn=ROLE_ARN,
 AlgorithmSpecification={
 'TrainingImage': ECR_URI,
 'TrainingInputMode': 'File',
 },
 InputDataConfig=[{
 'ChannelName': 'model',
 'DataSource': {'S3DataSource': {
 'S3DataType': 'S3Prefix',
 'S3Uri': ANYMAL_MODEL_S3,
 'S3DataDistributionType': 'FullyReplicated',
 }},
 }],
 OutputDataConfig={'S3OutputPath': f's3://{BUCKET}/isaac-lab/videos/'},
 ResourceConfig={
 'InstanceType': 'ml.g5.xlarge',
 'InstanceCount': 1,
 'VolumeSizeInGB': 100,
 },
 StoppingCondition={'MaxRuntimeInSeconds': 1800},
 HyperParameters={
 'mode': 'play',
 'task': 'Isaac-Velocity-Flat-Anymal-D-v0',
 'framework': 'rsl_rl',
 'video_length': '400',
 },
)

print(f' Anymal-D video render launched: {ANYMAL_VIDEO_JOB}')
print(f' ~5-10 min. MP4 will be inside the output artifact.')
print()
print('Download when complete:')
print(f' aws s3 cp s3://{BUCKET}/isaac-lab/videos/{ANYMAL_VIDEO_JOB}/output/model.tar.gz /tmp/anymal.tar.gz')
print(f' tar -xzf /tmp/anymal.tar.gz -C /tmp/anymal && ls /tmp/anymal/videos/')


## 6 — Render UR10 Arm Video

Renders an MP4 of the UR10 arm reaching policy.

**Requires:** Section 2 completed (`UR10_MODEL_S3` set).

In [ ]:
# Requires: UR10_MODEL_S3 from Section 2 poll cell
UR10_VIDEO_JOB = f'isaac-lab-video-ur10-{int(time.time())}'

sm.create_training_job(
 TrainingJobName=UR10_VIDEO_JOB,
 RoleArn=ROLE_ARN,
 AlgorithmSpecification={
 'TrainingImage': ECR_URI,
 'TrainingInputMode': 'File',
 },
 InputDataConfig=[{
 'ChannelName': 'model',
 'DataSource': {'S3DataSource': {
 'S3DataType': 'S3Prefix',
 'S3Uri': UR10_MODEL_S3,
 'S3DataDistributionType': 'FullyReplicated',
 }},
 }],
 OutputDataConfig={'S3OutputPath': f's3://{BUCKET}/isaac-lab/videos/'},
 ResourceConfig={
 'InstanceType': 'ml.g5.xlarge',
 'InstanceCount': 1,
 'VolumeSizeInGB': 100,
 },
 StoppingCondition={'MaxRuntimeInSeconds': 1800},
 HyperParameters={
 'mode': 'play',
 'task': 'Isaac-Reach-UR10-v0',
 'framework': 'rsl_rl',
 'video_length': '400',
 },
)

print(f' UR10 video render launched: {UR10_VIDEO_JOB}')
print(f' ~5-10 min. MP4 will be inside the output artifact.')
print()
print('Download when complete:')
print(f' aws s3 cp s3://{BUCKET}/isaac-lab/videos/{UR10_VIDEO_JOB}/output/model.tar.gz /tmp/ur10.tar.gz')
print(f' tar -xzf /tmp/ur10.tar.gz -C /tmp/ur10 && ls /tmp/ur10/videos/')


## 7 — Summary

### What's working

| Task | Status | Notes |
|------|--------|-------|
| `Isaac-Velocity-Flat-Anymal-D-v0` | Validated | 128 envs, reward -0.36→+8.58, 3741 steps/s |
| `Isaac-Reach-UR10-v0` | Validated | 4096 envs, ~10 min on ml.g5.xlarge |
| Container build | Validated | 11 min, 15.8 GB in ECR |
| Video rendering | Validated | Play mode works via SageMaker training job |

### What's future work

| Task | Status | Blocking issue |
|------|--------|----------------|
| `PickAndPlaceUR3-v0` | Not yet wired | Task is gym-registered but container entrypoint doesn't load custom envs |
| GR00T→MLP bridge | Not yet wired | `groot_to_rl_bridge.py` written; needs GR00T endpoint inference wired up |
| Multi-node training | Unvalidated | `--instance-count N` is wired but NCCL not tested on hardware |

### Next steps

1. Wire `PickAndPlaceUR3-v0` into `sm-train-entrypoint.sh` (load custom envs before
 calling Isaac Lab's train.py)
2. Validate the UR3 environment visually using Lab 2 workstation
3. Wire GR00T endpoint inference into the MLP behavioral cloning bridge

See the README.md for full context on next steps.